# Binary P-x-y Diagram

This notebook demonstrates how to generate a P-x-y diagram for a binary mixture at fixed temperature.

We calculate the bubble point (liquid) and dew point (vapor) curves.

## 1. Inputs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import chemthermo as ct

# --- Mixture Definition ---
component_names = ("Propane", "n-Butane")
temperature_K = 300.0  # 300 K (approx 27 C)
n_points = 25

components = tuple(ct.Component.from_database(name) for name in component_names)
eos = ct.PengRobinsonEOS()

print(f"System: {component_names} at {temperature_K} K")

## 2. Compute Phase Envelope

We iterate through composition space (x1 from 0 to 1) and calculate the saturation pressures.

In [ ]:
x1_range = np.linspace(0.0, 1.0, n_points)

bubble_p = []
dew_p = []

print("Computing phase envelope...")

for x1 in x1_range:
    x2 = 1.0 - x1
    z = (x1, x2)
    
    mix = ct.Mixture(components=components, composition=ct.Composition(fractions=z))
    
    # Bubble Point Pressure
    # P_bub is the pressure where liquid z boils.
    P_bub, y_eq = ct.bubble_pressure(mix, temperature_K, eos)
    bubble_p.append(P_bub)
    
    # Dew Point Pressure
    # P_dew is the pressure where vapor z condenses.
    P_dew, x_eq = ct.dew_pressure(mix, temperature_K, eos)
    dew_p.append(P_dew)

print("Done.")

## 3. Plot Diagram

In [ ]:
plt.figure(figsize=(8, 6))

# Convert Pa to bar for plotting
bubble_bar = np.array(bubble_p) / 1e5
dew_bar = np.array(dew_p) / 1e5

# P-x-y plots Pressure vs x (Bubble) and Pressure vs y (Dew) but we iterated SYSTEM composition z.
# For Bubble Point calc: z is Liquid Composition (x). result P_bub.
# For Dew Point calc: z is Vapor Composition (y). result P_dew.
# So we plot (z, P_bub) for Bubble Curve.
# And (z, P_dew) for Dew Curve.

plt.plot(x1_range, bubble_bar, 'b.-', label='Bubble Point (Liquid)')
plt.plot(x1_range, dew_bar, 'r.-', label='Dew Point (Vapor)')

plt.xlabel(f"Mole Fraction {component_names[0]}")
plt.ylabel("Pressure [bar]")
plt.title(f"P-x-y Diagram: {component_names[0]} + {component_names[1]} at {temperature_K} K")
plt.grid(True, alpha=0.3)
plt.legend()

plt.show()

## 4. Customization & Export

To customize this notebook:
1. Change `component_names` to other species.
2. Change `temperature_K`.

### Export Data to CSV

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "x1": x1_range,
    "P_bubble_bar": bubble_bar,
    "P_dew_bar": dew_bar
})

print("Data Preview:")
print(df.head())

# df.to_csv("pxy_data.csv", index=False)
